In [ ]:
import pandas as pd
from sklearn.linear_model import Lasso
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path
import random
from itertools import product
import optuna
import numpy as np
from scipy.stats import norm
import statsmodels.api as sm

from stage1 import lasso_rolling_window, calculate_r_squared
from stage2 import estimate_kappa_curve_fit, compute_alm_returns, compute_stage2_r_squared
from grid_search import grid_search, estimate_single_config

In [ ]:
# load feature matrix and response variable
feature_matrix = pd.read_csv("../../data/merged_return_topic_data.csv", index_col=0, parse_dates=True)
response_variables = pd.read_csv("../../data/response.csv", index_col=0, parse_dates=True)

In [ ]:
def to_ar1_innovations(X: pd.DataFrame, min_obs: int = 30) -> pd.DataFrame:
    """Return AR(1) innovations (residuals) for each column of X."""
    X_innov = pd.DataFrame(index=X.index, columns=X.columns, dtype="float64")

    for col in X.columns:
        s = pd.to_numeric(X[col], errors="coerce")
        tmp = pd.DataFrame({"x": s, "x_lag1": s.shift(1)}).dropna()
        if len(tmp) < min_obs or tmp["x"].nunique() < 3 or tmp["x_lag1"].nunique() < 3:
            continue

        res = sm.OLS(tmp["x"], sm.add_constant(tmp["x_lag1"])).fit()
        X_innov.loc[tmp.index, col] = res.resid

    return X_innov

In [ ]:
# set seed
random.seed(42)

# load feature matrix and response variable
feature_matrix = pd.read_csv("../../data/merged_return_topic_data.csv", index_col=0, parse_dates=True)
response_variables = pd.read_csv("../../data/response.csv", index_col=0, parse_dates=True)

# select a response variable (either marekt return or sp500 return)
y = response_variables['sprtrn']  # or 'sprtrn' for SP500 returns or vwretx

# transform returns to log
y = np.log(y+1)

# create feature matrix
X = feature_matrix.copy()

# Separate topic (name) and stock (numeric) columns
topic_cols = [col for col in X.columns if not str(col).isdigit()]
stock_cols = [col for col in X.columns if str(col).isdigit()]

# Final filtered dataframe
X = X[topic_cols]

# # usage
X = to_ar1_innovations(X)

# remove the first row wiht iloc
X = X.iloc[1:]

# ensure that all indices align
common_index = X.index.intersection(y.index)
X = X.loc[common_index]
y = y.loc[common_index]

## Grid Search

In [ ]:
import numpy as np
import pandas as pd

# -------------------------------------------------
# Objective function: maximize R^2 only
# -------------------------------------------------
def objective_function(row, r2_col="r2_insample_stage2"):
    r2 = pd.to_numeric(row.get(r2_col, np.nan), errors="coerce")
    if not np.isfinite(r2):
        return -1.0
    return r2

# -------------------------------------------------
# Single grid search
# -------------------------------------------------
param_grid = {
    "window_sizes": [36, 52, 78, 104, 150, 200],
    "n_lags": [1, 4, 8, 12],
    "lambdas": [0.0001, 0.00001],
}

# Run grid search once
summary_df, details_df = grid_search(X, y, param_grid, verbose=True)

# Compute objective
summary_df["objective"] = summary_df.apply(objective_function, axis=1)

# -------------------------------------------------
# Select best configuration
# -------------------------------------------------
valid = summary_df[summary_df["objective"] > -1.0]

if valid.empty:
    best_overall = None
    print("No valid configurations found.")
else:
    best_overall = valid.loc[valid["objective"].idxmax()]
    print(best_overall[["window_size", "n_lags", "lambda", "objective"]])

# -------------------------------------------------
# Return results
# -------------------------------------------------
summary_df, details_df, best_overall


Testing 48 configurations...


Grid search: 100%|██████████| 48/48 [06:19<00:00,  7.92s/it]


In [ ]:
# import numpy as np
# import pandas as pd

# # -------------------------------------------------
# # Objective function: maximize R^2 only
# # -------------------------------------------------
# def objective_function(row, r2_col="r2_insample_stage2"):
#     r2 = pd.to_numeric(row.get(r2_col, np.nan), errors="coerce")
#     if not np.isfinite(r2):
#         return -1.0
#     return r2


# # -------------------------------------------------
# # Iterative grid refinement
# # -------------------------------------------------
# results_all = []
# prev_best = -np.inf

# param_grid = {
#     "window_sizes": [36, 52, 78, 104, 150, 200],
#     "n_lags": [1, 4, 8, 12],
#     "lambdas": [0.0001, 0.00001],
# }

# for iteration in range(5):

#     # Run grid search
#     summary_df, _ = grid_search(X, y, param_grid, verbose=True)

#     # Compute objective
#     summary_df["objective"] = summary_df.apply(objective_function, axis=1)
#     results_all.append(summary_df)

#     # Check for valid candidates
#     valid_scores = summary_df.loc[summary_df["objective"] > -1.0, "objective"]
#     if valid_scores.empty:
#         break

#     best_idx = valid_scores.idxmax()
#     best_obj = valid_scores.max()

#     # Convergence check
#     if iteration > 0 and (best_obj - prev_best) < 1e-6:
#         break

#     prev_best = best_obj
#     best = summary_df.loc[best_idx]

#     # -------------------------------------------------
#     # Refine grid around best point
#     # -------------------------------------------------
#     w = int(best["window_size"])
#     l = int(best["n_lags"])
#     lam = float(best["lambda"])

#     param_grid = {
#         "window_sizes": sorted(
#             {int(w * f) for f in [0.75, 0.9, 1.0, 1.1, 1.25] if w * f >= 20}
#         ),
#         "n_lags": sorted({max(1, l - 2), l - 1, l, l + 1, l + 2}),
#         "lambdas": lam * np.array([0.5, 0.75, 1.0, 1.25, 1.5]),
#     }


# # -------------------------------------------------
# # Select best overall configuration
# # -------------------------------------------------
# final = pd.concat(results_all, ignore_index=True)
# final["objective"] = final.apply(objective_function, axis=1)

# best_overall = final.loc[final["objective"].idxmax()]
# print(best_overall[["window_size", "n_lags", "lambda", "objective"]])


## Bayesian Optimization with optuna

For each hyperparameter triplet we do the following:

    - run stage1 and stage2
    - collect r2 in-sample stage2 and kappa
    - compute objective function: r2*kappa
    - let optuna try 150 random/learned samples to maximize the objective function

In [ ]:
import numpy as np
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

# -----------------------------
# Focus region for economic meaning
# -----------------------------
TSTAT_COL = "kappa_tstat"
TSTAT_MIN = 1.96
TSTAT_MAX = 100.0

# Scoring / penalties (tune if needed)
SOFT_PENALTY_OUTSIDE = 2.0   # larger -> harder push into band
HARD_PRUNE_TOO_LARGE = True  # prune t-stat > TSTAT_MAX (often indicates instability)
EPS = 1e-12


def objective(trial):
    # Search space (keep yours; adjust if you like)
    window_size = trial.suggest_int("window_size", 30, 300, step=10)
    n_lags = trial.suggest_int("n_lags", 1, 10)
    lam = trial.suggest_float("lambda", 1e-6, 1e-3, log=True)

    res = estimate_single_config(
        X, y, window_size, n_lags, lam,
        standardize=True, verbose=False, return_details=False
    )
    summary = (res or {}).get("summary", {})

    r2 = float(summary.get("r2_insample_stage2", np.nan))
    kappa = float(summary.get("kappa", np.nan))
    t = float(summary.get(TSTAT_COL, np.nan))

    # Store diagnostics
    trial.set_user_attr("r2_raw", r2)
    trial.set_user_attr("kappa_raw", kappa)
    trial.set_user_attr("kappa_tstat", t)

    # Basic validity
    if not (np.isfinite(r2) and np.isfinite(kappa) and np.isfinite(t)):
        raise optuna.TrialPruned()
    if r2 < 0 or kappa <= 0:
        # guide away without blowing up the optimizer
        return -0.1 - 0.01 * abs(r2) - 0.01 * abs(kappa)

    base = float(np.sqrt(max(r2, 0.0) * max(kappa, EPS)))
    trial.set_user_attr("base_score", base)

    # -----------------------------
    # Enforce / encourage the economically meaningful region:
    #   1.96 < t < 100
    # -----------------------------

    # If t-stat is *absurdly* large, treat as pathological and prune.
    if HARD_PRUNE_TOO_LARGE and t > TSTAT_MAX:
        trial.set_user_attr("t_band_status", "pruned_too_large")
        raise optuna.TrialPruned()

    # If not significant, keep trial but strongly penalize (so TPE learns)
    if t < TSTAT_MIN:
        # distance below the threshold, scaled to be comparable across magnitudes
        dist = (TSTAT_MIN - t) / TSTAT_MIN  # in (0, +inf)
        penalty = SOFT_PENALTY_OUTSIDE * dist
        score = base / (1.0 + penalty)
        trial.set_user_attr("t_band_status", "below_min_soft_penalty")
        trial.set_user_attr("penalty", penalty)
        return float(score)

    # In band -> best region, no penalty
    trial.set_user_attr("t_band_status", "in_band")
    trial.set_user_attr("penalty", 0.0)
    return base


# -----------------------------
# Sampler + pruner
# -----------------------------
sampler = TPESampler(
    seed=42,
    n_startup_trials=30,
    multivariate=True,
    constant_liar=True
)

pruner = MedianPruner(n_startup_trials=20, n_warmup_steps=5)

study = optuna.create_study(direction="maximize", sampler=sampler, pruner=pruner)


def print_best_callback(study, trial):
    if trial.number % 10 == 0 and study.best_trial is not None:
        bt = study.best_trial
        print(
            f"Trial {trial.number}: best={study.best_value:.4f} "
            f"(t={bt.user_attrs.get('kappa_tstat', np.nan):.2f}, "
            f"R2={bt.user_attrs.get('r2_raw', np.nan):.4f}, "
            f"kappa={bt.user_attrs.get('kappa_raw', np.nan):.4f})"
        )


study.optimize(
    objective,
    n_trials=250,
    n_jobs=6,
    callbacks=[print_best_callback],
    show_progress_bar=True
)

print("\n" + "=" * 60)
print("OPTIMIZATION RESULTS")
print("=" * 60)
print(f"Best Score (objective): {study.best_value:.6f}")
print(f"Best Params: {study.best_params}")

best_trial = study.best_trial
print("\nBest Trial Metrics:")
print(f"  R²:      {best_trial.user_attrs.get('r2_raw', np.nan):.6f}")
print(f"  Kappa:   {best_trial.user_attrs.get('kappa_raw', np.nan):.6f}")
print(f"  t-stat:  {best_trial.user_attrs.get('kappa_tstat', np.nan):.6f}")
print(f"  Base:    {best_trial.user_attrs.get('base_score', np.nan):.6f}")
print(f"  Status:  {best_trial.user_attrs.get('t_band_status', '')}")

# -----------------------------
# Reporting: "best within band" explicitly
# -----------------------------
in_band = [
    t for t in study.trials
    if t.value is not None
    and t.state == optuna.trial.TrialState.COMPLETE
    and (t.user_attrs.get("kappa_tstat", -np.inf) >= TSTAT_MIN)
    and (t.user_attrs.get("kappa_tstat", np.inf) <= TSTAT_MAX)
]

in_band.sort(key=lambda tr: tr.user_attrs.get("base_score", -np.inf), reverse=True)

print("\n" + "=" * 60)
print(f"TOP 10 TRIALS IN ECONOMIC BAND ({TSTAT_MIN}–{TSTAT_MAX})")
print("=" * 60)
if in_band:
    for i, tr in enumerate(in_band[:10], 1):
        print(
            f"{i}. base={tr.user_attrs.get('base_score', np.nan):.6f} | "
            f"obj={tr.value:.6f} | "
            f"R²={tr.user_attrs.get('r2_raw', np.nan):.4f} | "
            f"kappa={tr.user_attrs.get('kappa_raw', np.nan):.4f} | "
            f"t={tr.user_attrs.get('kappa_tstat', np.nan):.2f} | "
            f"params={tr.params}"
        )
else:
    print("No completed trials ended up in the target t-stat band.")

# -----------------------------
# Diagnostics: pruned counts & where mass is
# -----------------------------
states = [tr.state for tr in study.trials]
n_pruned = sum(s == optuna.trial.TrialState.PRUNED for s in states)
n_complete = sum(s == optuna.trial.TrialState.COMPLETE for s in states)

below = sum(
    (tr.state == optuna.trial.TrialState.COMPLETE)
    and (tr.user_attrs.get("kappa_tstat", -np.inf) < TSTAT_MIN)
    for tr in study.trials
)
above_pruned = sum(
    (tr.state == optuna.trial.TrialState.PRUNED)
    and (tr.user_attrs.get("t_band_status", "") == "pruned_too_large")
    for tr in study.trials
)

print("\n" + "=" * 60)
print("DIAGNOSTICS")
print("=" * 60)
print(f"Complete trials: {n_complete}/{len(study.trials)}")
print(f"Pruned trials:   {n_pruned}/{len(study.trials)}")
print(f"Complete but t < {TSTAT_MIN}: {below}")
print(f"Pruned for t > {TSTAT_MAX}:  {above_pruned}")


In [ ]:
res  = estimate_single_config(
    X, y,
    window_size= 10,
    n_lags=5,
    lambda_val=0.0025995,
    standardize=True,   
    verbose=True,
    return_details=True
)

In [ ]:
res